[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/03_comparison.ipynb)


# R1-Q2 Week 4 — Compare hubs to known regulators

You arrive at this notebook with two per-tissue hub lists (212 shoot
hubs, 467 root hubs) from Notebook 2. The question this notebook
answers is: **are those hubs enriched for known stress-regulating
transcription factors?**

The reference set is Hakkak & Tohidfar (2026) Supplementary Table 3 —
roughly 60 TFs across 15 families, all identified from a stress-response
meta-analysis. If a co-expression network really does surface
biologically meaningful regulators, the hubs and the reference should
overlap more than chance would predict.

By the end of this notebook you will have:

- Per-tissue hub tables annotated against the Hakkak reference, saved
  as `shoot_compare.parquet` and `root_compare.parquet`.
- A formal enrichment test (one-sided hypergeometric, Bonferroni-
  corrected) for each tissue.
- A descriptive look at the 66 cross-tissue hubs against the reference.
- A summary JSON capturing the results for use in the writeup.
- The answer to R1-Q2's question, in your own words, written as the
  Section 6 synthesis.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026, point to the R1-Q2 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 1) Setup and load inputs

Three pieces of input feed the rest of this notebook:

1. **The per-tissue hub tables from N2** (`shoot_hubs.parquet`,
   `root_hubs.parquet`). Each row is a probe identified as a hub in
   a stress-relevant module.
2. **The reference set** (`hakkak_2026_supp3.csv`) — a curated list
   of stress-responsive transcription factors from Hakkak & Tohidfar
   (2026). The file is uploaded to the same R1-Q2 output directory
   that N2 wrote to; if you haven't put it there yet, see the load
   cell below for download instructions.
3. **The pre-commits** (`precommit.json`) — your Week 1 lock-ins for
   the reference set, the statistical framework, and the background
   gene set. We echo the three entries that this notebook acts on so
   the locked decisions are visible while you read the analysis.

Each load below has a defensive error path. If a file isn't found,
the error tells you what's missing and how to produce it.

### 1.1) Hub tables from N2

Two Parquet files, one per tissue. Each row is a probe (the index),
with columns capturing module membership, the probe's kME value
within that module, and the stresses the module is associated with.

In [ ]:
import pandas as pd

shoot_hubs_path = OUTPUT_DIR / "shoot_hubs.parquet"
root_hubs_path = OUTPUT_DIR / "root_hubs.parquet"

try:
    shoot_hubs = pd.read_parquet(shoot_hubs_path)
    root_hubs = pd.read_parquet(root_hubs_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find an N2 hub table.\n"
        f"  Looked for: {shoot_hubs_path}\n"
        f"              {root_hubs_path}\n\n"
        f"Run `02_hub_identification.ipynb` to completion first — "
        f"its Section 5 writes both Parquets to this location.\n"
    ) from None

print(f"Shoot hubs: {len(shoot_hubs):>4} probes")
print(f"Root hubs:  {len(root_hubs):>4} probes")
print()
print(f"Columns:     {shoot_hubs.columns.tolist()}")
print(f"Index name:  {shoot_hubs.index.name!r}")
print()
print("Shoot head:")
print(shoot_hubs.head())

### 1.2) Reference set (Hakkak & Tohidfar 2026 Supplementary Table 3)

The reference file is a curated list of transcription factors that
Hakkak & Tohidfar identified as differentially expressed across a
combined drought/salt meta-analysis. Each row records:

- `ORF` — the AGI gene identifier (the column is called `ORF` in the
  source file, not `agi` or `gene_id`; this is a Hakkak convention).
- `Symbol` — the canonical gene symbol (e.g., `DREB2A`).
- `Family` — the transcription factor family (e.g., `ERF`, `MYB`).
- `Group` — `Up-regulated gene` or `Down-regulated gene`.

The source file has a layout quirk: it's published as a
two-column-block CSV, with up-regulated TFs in the left block and
down-regulated TFs in the right block, separated by a blank column.
The load cell collapses these into one tidy table.

Two more quirks to be aware of, both handled below:

1. **The `ORF` column is in mixed case** (`At1g01720` rather than
   `AT1G01720`). The probe-to-AGI mapping in Section 2 normalizes
   to upper case, so we apply the same normalization here at load
   time. Otherwise joins downstream would silently miss everything.
2. **One row is duplicated.** `AT2G47190` appears twice — once with
   family `ERF`, once with family `MYB`. Same gene, conflicting
   family label in the source. The defensive check below surfaces
   any duplicates so you can see them; we carry both rows in the
   loaded table for fidelity to source, and handle the dedup
   downstream where it matters (Section 3, the per-family
   breakdown).

If the file isn't present, download it from the journal's
supplementary materials. Springer publishes it as
`10709_2026_261_MOESM1_ESM.csv` (despite our internal name being
`...supp3...` — the file numbering on the journal site doesn't match
the paper's narrative table numbering). Rename and upload to
`OUTPUT_DIR`.

In [ ]:
hakkak_path = OUTPUT_DIR / "hakkak_2026_supp3.csv"

try:
    raw = pd.read_csv(hakkak_path, skiprows=1)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find the Hakkak reference set at:\n"
        f"  {hakkak_path}\n\n"
        f"Download `10709_2026_261_MOESM1_ESM.csv` from the journal's "
        f"supplementary materials for Hakkak & Tohidfar (2026), rename "
        f"it to `hakkak_2026_supp3.csv`, and place it at the path above.\n"
    ) from None

# The file's two-column-block layout: columns 0-3 are up-regulated TFs,
# column 4 is a blank separator, columns 5-8 are down-regulated TFs.
left = raw.iloc[:, [0, 1, 2, 3]].copy()
right = raw.iloc[:, [5, 6, 7, 8]].copy()
right.columns = left.columns

hakkak_tf = (
    pd.concat([left, right], ignore_index=True)
    .dropna(how='all')
    .reset_index(drop=True)
)

# Normalize case on the AGI column; strip any stray whitespace.
hakkak_tf['ORF'] = hakkak_tf['ORF'].astype(str).str.strip().str.upper()

# Defensive: drop any non-AGI rows (e.g., trailing footer rows).
hakkak_tf = hakkak_tf[hakkak_tf['ORF'].str.startswith('AT')].reset_index(drop=True)

print(f"Reference rows:    {len(hakkak_tf)}")
print(f"Unique AGIs:       {hakkak_tf['ORF'].nunique()}")
print(f"Unique families:   {hakkak_tf['Family'].nunique()}")
print()
print("Family breakdown:")
print(hakkak_tf['Family'].value_counts().to_string())

In [ ]:
# The source file is known to carry one duplicate AGI with conflicting
# family labels. Print any duplicates so the student sees them; we
# carry the rows in `hakkak_tf` and handle the dedup downstream where
# it matters.
duplicates = (
    hakkak_tf[hakkak_tf['ORF'].duplicated(keep=False)]
    .sort_values('ORF')
)

if len(duplicates) > 0:
    print(f"Duplicate AGIs in the reference set ({len(duplicates)} rows):")
    print(duplicates.to_string(index=False))
    print()
    print("Both rows are carried in `hakkak_tf` for fidelity to source.")
    print("Section 3's per-family breakdown deduplicates by uppercase ORF.")
else:
    print("No duplicate AGIs in the reference set.")

### 1.3) Pre-commit echo

Three of the six pre-commits you locked in N0 inform what this
notebook does:

- `reference_set` — which file to compare against, and which column
  holds the AGI codes.
- `statistical_framework` — which test to run, with how many tests
  and what correction.
- `background_gene_set` — the universe of probes that could have
  been hubs, per tissue. Section 4's hypergeometric test uses these
  counts as its `N` (population size) parameter.

The cell below loads `precommit.json` and prints these three blocks
in full. Cross-check the values against what you remember locking in
Week 1; if anything looks wrong, stop and re-open N0 before running
the rest of the notebook.

In [ ]:
import json

precommit_path = OUTPUT_DIR / "precommit.json"
with open(precommit_path) as f:
    precommit = json.load(f)

for category in ("reference_set", "statistical_framework", "background_gene_set"):
    print(f"--- {category} ---")
    print(json.dumps(precommit[category], indent=2))
    print()

With the three inputs loaded and the locked decisions in view, the
foreground (hubs) and reference (Hakkak's TF table) are in hand —
but in different identifier spaces. The hub tables are keyed by
Affymetrix probe ID (`249264_s_at`); Hakkak's table is keyed by AGI
gene identifier (`AT5G42570`). Section 2 bridges that gap with a
single library call.